# Balanced GELU normalization stability sweep, total batch size 128

This notebook tests four local UniversalXAS heads with fixed exported 64D E2E FEFF features and canonical 141D targets. It does not retrain the encoder.

The balanced sampler uses 16 rows from each of the eight elements in every batch. The sweep trains 12 candidates: four normalizations across seeds 44, 45, and 46. Validation selects one normalization by the mean of its three per-seed mean validation eta values. After selection, the notebook evaluates all 12 candidates on the test split. Test metrics do not select the model.


## 1. Imports and fixed configuration

Set `OMNIXAS_E2E_UNIVERSAL_RUN` to the E2E UniversalXAS source run when the default path is not valid. The output directory is distinct from the single-seed batch-size sweep.


In [1]:
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import dataclass
from hashlib import sha256
import json
import math
import os
from pathlib import Path
from typing import Any

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, Sampler, TensorDataset
from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import PlModule

torch.set_float32_matmul_precision("medium")
ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
FEFF_DATASETS = [f"{element}_FEFF" for element in ELEMENTS]
INPUT_DIM, OUTPUT_DIM = 64, 141
HIDDEN_DIMS = [500, 500, 550]
SEEDS = [44, 45, 46]
DROPOUT = 0.10
LEARNING_RATE = 5e-4
TOTAL_BATCH_SIZE = 128
ROWS_PER_ELEMENT = 16
NORMALIZATIONS = ["BatchNorm1d", "LayerNorm", "GroupNorm", "None"]
GROUP_NORM_GROUPS = 2
SCHEDULER = "ReduceLROnPlateau"
SCHEDULER_MODE, SCHEDULER_FACTOR = "min", 0.5
BASE_SCHEDULER_PATIENCE, SCHEDULER_FREQUENCY = 8, 2
MIN_LR = 1e-6
MAX_EPOCHS, BASE_EARLY_STOPPING_PATIENCE = 800, 60
CHECK_VAL_EVERY_N_EPOCH = 2
MONITOR = "val_median_mse"
EXPECTED_CANDIDATES = 12
EXPECTED_DETAILED_ROWS = 96
EXPECTED_NORMALIZATION_ROWS = 4
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def locate_repo_root(start: Path) -> Path:
    configured = os.environ.get("OMNIXAS_REPO_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
        raise FileNotFoundError(f"OMNIXAS_REPO_ROOT is not an OmniXAS repository: {candidate}")
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OmniXAS repository")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_value(value: Any) -> Any:
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    return value


def fingerprint(value: Any) -> str:
    payload = json.dumps(json_value(value), sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()


REPO_ROOT = locate_repo_root(Path.cwd().resolve())
configured_source = os.environ.get("OMNIXAS_E2E_UNIVERSAL_RUN")
SOURCE_RUN = Path(configured_source).expanduser().resolve() if configured_source else (REPO_ROOT.parent / "fulltrainingcopy072726" / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42").resolve()
FEATURES_DIR = SOURCE_RUN / "features"
ML_DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
MATERIAL_IDS_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
RUN_DIR = SOURCE_RUN / "sweep_balanced_gelu_normalizations_batch128_seeds44_45_46"
for required in (SOURCE_RUN, FEATURES_DIR, ML_DATA_DIR, MATERIAL_IDS_DIR):
    if not required.exists():
        raise FileNotFoundError(f"Missing required path: {required}")
print(f"Source run: {SOURCE_RUN}")
print(f"Output run: {RUN_DIR}")
print(f"Device: {DEVICE}")


Source run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
Output run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_normalizations_batch128_seeds44_45_46
Device: cuda:0


## 2. Load and validate fixed exported data

The checks verify feature and target dimensions, finite values, exact exported-to-canonical target equality, ID alignment, duplicate IDs, valid site suffixes, and material isolation across splits.


In [2]:
@dataclass(frozen=True)
class LoadedSplit:
    split: MLSplits
    ids: dict[str, list[str]]
    materials: dict[str, list[str]]


def read_matrix(path: Path) -> np.ndarray:
    return np.atleast_2d(np.loadtxt(path, dtype=np.float32))


def read_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def load_dataset(dataset: str) -> LoadedSplit:
    parts: dict[str, MLData] = {}
    ids_by_split: dict[str, list[str]] = {}
    materials_by_split: dict[str, list[str]] = {}
    seen_ids: set[str] = set()
    for split_name in ("train", "val", "test"):
        paths = {
            "X": FEATURES_DIR / f"{dataset}_{split_name}_X.txt",
            "exported_y": FEATURES_DIR / f"{dataset}_{split_name}_y.txt",
            "canonical_y": ML_DATA_DIR / f"{dataset}_{split_name}_y.txt",
            "ids": MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt",
        }
        for path in paths.values():
            if not path.is_file():
                raise FileNotFoundError(f"Missing {dataset} {split_name} file: {path}")
        X, exported_y, canonical_y = (read_matrix(paths[key]) for key in ("X", "exported_y", "canonical_y"))
        if X.ndim != 2 or X.shape[1] != INPUT_DIM or exported_y.ndim != 2 or exported_y.shape[1] != OUTPUT_DIM:
            raise ValueError(f"Invalid dimensions for {dataset} {split_name}: X={X.shape}, y={exported_y.shape}")
        if canonical_y.shape != exported_y.shape or X.shape[0] != exported_y.shape[0]:
            raise ValueError(f"Row or target dimensions do not match for {dataset} {split_name}")
        if not np.isfinite(X).all() or not np.isfinite(exported_y).all() or not np.isfinite(canonical_y).all():
            raise ValueError(f"Non-finite data in {dataset} {split_name}")
        if not np.array_equal(exported_y, canonical_y):
            raise ValueError(f"Exported targets do not exactly match canonical targets for {dataset} {split_name}")
        ids = read_ids(paths["ids"])
        if len(ids) != len(X) or len(set(ids)) != len(ids):
            raise ValueError(f"ID count or duplicate check failed for {dataset} {split_name}")
        overlap = seen_ids.intersection(ids)
        if overlap:
            raise ValueError(f"Duplicate full IDs across splits for {dataset}: {sorted(overlap)[:5]}")
        seen_ids.update(ids)
        materials = []
        for row in ids:
            material, separator, site = row.rpartition("_")
            if not separator or not material or not site.isascii() or not site.isdigit() or int(site) < 0:
                raise ValueError(f"Invalid material/site ID in {paths['ids']}: {row!r}")
            materials.append(material)
        ids_by_split[split_name], materials_by_split[split_name] = ids, materials
        parts[split_name] = MLData(X=X, y=exported_y)
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = set(materials_by_split[left]).intersection(materials_by_split[right])
        if overlap:
            raise ValueError(f"Material split isolation failed for {dataset} {left}/{right}: {sorted(overlap)[:5]}")
    return LoadedSplit(MLSplits(**parts), ids_by_split, materials_by_split)


loaded = {dataset: load_dataset(dataset) for dataset in FEFF_DATASETS}
splits = {dataset: item.split for dataset, item in loaded.items()}
source_files = sorted({path for dataset in FEFF_DATASETS for split_name in ("train", "val", "test") for path in (
    FEATURES_DIR / f"{dataset}_{split_name}_X.txt", FEATURES_DIR / f"{dataset}_{split_name}_y.txt",
    ML_DATA_DIR / f"{dataset}_{split_name}_y.txt", MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt")})
print({dataset: {name: len(getattr(split, name)) for name in ("train", "val", "test")} for dataset, split in splits.items()})


{'Ti_FEFF': {'train': 5140, 'val': 641, 'test': 641}, 'V_FEFF': {'train': 8653, 'val': 1080, 'test': 1080}, 'Cr_FEFF': {'train': 2457, 'val': 305, 'test': 305}, 'Mn_FEFF': {'train': 13644, 'val': 1704, 'test': 1704}, 'Fe_FEFF': {'train': 9657, 'val': 1205, 'test': 1205}, 'Co_FEFF': {'train': 8605, 'val': 1074, 'test': 1074}, 'Ni_FEFF': {'train': 3471, 'val': 432, 'test': 432}, 'Cu_FEFF': {'train': 3340, 'val': 416, 'test': 416}}


## 3. Local GELU heads and exact balanced sampler

The hidden block is `Linear -> normalization -> GELU -> Dropout`. The `None` candidate omits normalization. The output block is `Linear -> Softplus`.

The sampler redraws without replacement from each element pool with `seed + epoch`. The self-check runs for all 12 candidates before training.


In [3]:
def make_normalization(name: str, width: int) -> nn.Module | None:
    if name == "BatchNorm1d":
        return nn.BatchNorm1d(width)
    if name == "LayerNorm":
        return nn.LayerNorm(width)
    if name == "GroupNorm":
        return nn.GroupNorm(GROUP_NORM_GROUPS, width)
    if name == "None":
        return None
    raise ValueError(f"Unsupported normalization {name!r}. Choose one of {NORMALIZATIONS}")


class GELUXASBlock(nn.Sequential):
    def __init__(self, input_dim: int, hidden_dims: list[int], output_dim: int, dropout: float, normalization: str):
        dims = [input_dim, *hidden_dims, output_dim]
        layers: list[nn.Module] = []
        for index, (width_in, width_out) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(width_in, width_out))
            if index < len(dims) - 2:
                norm = make_normalization(normalization, width_out)
                if norm is not None:
                    layers.append(norm)
                layers.extend([nn.GELU(), nn.Dropout(dropout)])
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


class ElementStratifiedBatchSampler(Sampler[list[int]]):
    def __init__(self, indices_by_element: dict[int, list[int]], per_element: int, seed: int):
        if set(indices_by_element) != set(range(len(ELEMENTS))) or any(not values for values in indices_by_element.values()):
            raise ValueError("Sampler requires one non-empty pool for each of the eight elements")
        if per_element < 1:
            raise ValueError("per_element must be positive")
        self.indices_by_element = {int(key): list(value) for key, value in indices_by_element.items()}
        self.element_order, self.per_element, self.seed, self.epoch = sorted(self.indices_by_element), int(per_element), int(seed), 0
        self.n_batches = min(map(len, self.indices_by_element.values())) // self.per_element
        if self.n_batches < 1:
            raise ValueError("Every element needs at least per_element rows")

    def __len__(self) -> int:
        return self.n_batches

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        selected = {element: rng.permutation(self.indices_by_element[element])[: self.n_batches * self.per_element] for element in self.element_order}
        batches = []
        for batch_number in range(self.n_batches):
            batch = np.concatenate([selected[element][batch_number * self.per_element:(batch_number + 1) * self.per_element] for element in self.element_order])
            rng.shuffle(batch)
            batches.append(batch.tolist())
        return iter(batches)


def combine_split(split_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_blocks, y_blocks, element_blocks = [], [], []
    for element_index, dataset in enumerate(FEFF_DATASETS):
        part = getattr(splits[dataset], split_name)
        X_blocks.append(part.X)
        y_blocks.append(part.y)
        element_blocks.append(np.full(len(part), element_index, dtype=np.int64))
    return np.concatenate(X_blocks), np.concatenate(y_blocks), np.concatenate(element_blocks)


universal_train_X, universal_train_y, train_element_codes = combine_split("train")
universal_val_X, universal_val_y, val_element_codes = combine_split("val")
train_indices_by_element = {i: np.flatnonzero(train_element_codes == i).tolist() for i in range(len(ELEMENTS))}
val_baselines = np.array([np.median(np.mean((splits[d].val.y - splits[d].train.y.mean(axis=0)) ** 2, axis=1)) for d in FEFF_DATASETS], dtype=np.float32)
candidate_specs = [
    {"candidate_id": f"normalization_{normalization}_seed_{seed}", "normalization": normalization, "seed": seed,
     "batch_size": TOTAL_BATCH_SIZE, "rows_per_element": ROWS_PER_ELEMENT}
    for normalization in NORMALIZATIONS for seed in SEEDS
]
if len(candidate_specs) != EXPECTED_CANDIDATES:
    raise AssertionError("Expected 12 normalization-seed candidates")

sampler_checks = []
for candidate in candidate_specs:
    sampler = ElementStratifiedBatchSampler(train_indices_by_element, candidate["rows_per_element"], candidate["seed"])
    batches = list(sampler)
    expected = min(map(len, train_indices_by_element.values())) // ROWS_PER_ELEMENT
    if len(batches) != expected:
        raise AssertionError(f"Expected {expected} batches, got {len(batches)}")
    sampled = []
    for batch in batches:
        if len(batch) != TOTAL_BATCH_SIZE:
            raise AssertionError(f"Wrong batch size: {len(batch)}")
        indices = np.asarray(batch, dtype=np.int64)
        if np.any(indices < 0) or np.any(indices >= len(universal_train_y)):
            raise AssertionError("Out-of-range sampled index")
        counts = np.bincount(train_element_codes[indices], minlength=len(ELEMENTS))
        if not np.array_equal(counts, np.full(len(ELEMENTS), ROWS_PER_ELEMENT)):
            raise AssertionError(f"Unequal element counts: {counts.tolist()}")
        sampled.extend(batch)
    if len(sampled) != len(set(sampled)):
        raise AssertionError("Duplicate row index in one epoch")
    full_batches = math.ceil(len(universal_train_y) / TOTAL_BATCH_SIZE)
    scale = full_batches / len(sampler)
    candidate.update({"full_data_batches_per_epoch": full_batches, "balanced_batches_per_epoch": len(sampler), "update_budget_scale": scale,
                      "effective_scheduler_patience": round(BASE_SCHEDULER_PATIENCE * scale), "effective_early_stopping_patience": round(BASE_EARLY_STOPPING_PATIENCE * scale)})
    sampler_checks.append({"candidate_id": candidate["candidate_id"], "normalization": candidate["normalization"], "seed": candidate["seed"], "batch_size": TOTAL_BATCH_SIZE, "rows_per_element": ROWS_PER_ELEMENT, "expected_batches": expected})
print("Sampler self-checks passed for 12 candidates")
display(pd.DataFrame(sampler_checks))


Sampler self-checks passed for 12 candidates


,candidate_id,normalization,seed,batch_size,rows_per_element,expected_batches
0,normalization_BatchNorm1d_seed_44,BatchNorm1d,44,128,16,153
1,normalization_BatchNorm1d_seed_45,BatchNorm1d,45,128,16,153
2,normalization_BatchNorm1d_seed_46,BatchNorm1d,46,128,16,153
3,normalization_LayerNorm_seed_44,LayerNorm,44,128,16,153
4,normalization_LayerNorm_seed_45,LayerNorm,45,128,16,153
5,normalization_LayerNorm_seed_46,LayerNorm,46,128,16,153
6,normalization_GroupNorm_seed_44,GroupNorm,44,128,16,153
7,normalization_GroupNorm_seed_45,GroupNorm,45,128,16,153
8,normalization_GroupNorm_seed_46,GroupNorm,46,128,16,153
9,normalization_None_seed_44,None,44,128,16,153


## 4. Matching run state and candidate settings

Reuse requires matching settings, a complete `TRAINING_COMPLETE.json`, exactly one best checkpoint, and a matching SHA256. A non-empty incomplete candidate directory fails clearly.


In [4]:
candidate_settings = []
for candidate in candidate_specs:
    normalization = candidate["normalization"]
    seed = candidate["seed"]
    hidden_block = "Linear -> GELU -> Dropout" if normalization == "None" else f"Linear -> {normalization} -> GELU -> Dropout"
    candidate_settings.append({
        "candidate_id": candidate["candidate_id"], "model": "UniversalXAS", "activation": "GELU", "normalization": normalization, "group_norm_groups": GROUP_NORM_GROUPS if normalization == "GroupNorm" else None,
        "architecture": "64 -> 500 -> 500 -> 550 -> 141", "hidden_block": hidden_block, "output_block": "Linear -> Softplus",
        "input_dim": INPUT_DIM, "hidden_dims": HIDDEN_DIMS, "output_dim": OUTPUT_DIM, "seed": seed, "dropout": DROPOUT,
        "optimizer": "Adam", "lr": LEARNING_RATE, "batch_size": TOTAL_BATCH_SIZE, "rows_per_element": ROWS_PER_ELEMENT,
        "sampler": "ElementStratifiedBatchSampler", "sampler_redraw": "without replacement per element pool with seed + epoch",
        "scheduler": SCHEDULER, "scheduler_mode": SCHEDULER_MODE, "scheduler_factor": SCHEDULER_FACTOR,
        "scheduler_base_patience": BASE_SCHEDULER_PATIENCE, "scheduler_effective_patience": candidate["effective_scheduler_patience"],
        "scheduler_frequency": SCHEDULER_FREQUENCY, "min_lr": MIN_LR, "max_epochs": MAX_EPOCHS,
        "early_stopping_base_patience": BASE_EARLY_STOPPING_PATIENCE, "early_stopping_effective_patience": candidate["effective_early_stopping_patience"],
        "validation_every_n_epochs": CHECK_VAL_EVERY_N_EPOCH, "monitor": MONITOR,
        "full_data_batches_per_epoch": candidate["full_data_batches_per_epoch"], "balanced_batches_per_epoch": candidate["balanced_batches_per_epoch"], "update_budget_scale": candidate["update_budget_scale"],
        "source_run": str(SOURCE_RUN), "feature_provenance": "fixed exported 64D E2E FEFF features", "encoder_retrained": False,
    })
settings_core = {
    "experiment": "balanced GELU normalization stability sweep, batch size 128, seeds 44 45 46", "source_run": str(SOURCE_RUN),
    "output_run": str(RUN_DIR), "seeds": SEEDS, "candidate_count": EXPECTED_CANDIDATES, "detailed_rows_per_split": EXPECTED_DETAILED_ROWS,
    "feature_provenance": "fixed exported 64D E2E FEFF features", "encoder_retrained": False,
    "architecture": {"input_dim": INPUT_DIM, "hidden_dims": HIDDEN_DIMS, "output_dim": OUTPUT_DIM, "output_block": "Linear -> Softplus", "normalization_candidates": NORMALIZATIONS, "group_norm_groups": GROUP_NORM_GROUPS},
    "candidate_settings": candidate_settings,
    "selection_policy": "Select one normalization by the mean of its three per-seed mean validation eta values across all eight FEFF datasets. Select the validation-best seed within that normalization for transparency. Never use test metrics for selection.",
    "learning_rate_policy": "Keep Adam learning rate fixed at 5e-4.",
}
settings_fingerprint = fingerprint(settings_core)
manifest_path = RUN_DIR / "settings_manifest.json"
if RUN_DIR.exists():
    if not RUN_DIR.is_dir():
        raise FileExistsError(f"Output path is not a directory: {RUN_DIR}")
    if any(RUN_DIR.iterdir()):
        if not manifest_path.is_file():
            raise RuntimeError(f"Non-empty run directory has no settings_manifest.json: {RUN_DIR}")
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        if existing.get("configuration_fingerprint") != settings_fingerprint:
            raise RuntimeError("Existing run settings do not match this notebook")
else:
    RUN_DIR.mkdir(parents=True)
manifest = {**settings_core, "configuration_fingerprint": settings_fingerprint, "status": "incomplete", "output_run": str(RUN_DIR),
            "source_files": [{"path": str(path), "sha256": sha256_file(path)} for path in source_files], "checkpoint_hashes_recorded": False}
manifest_path.write_text(json.dumps(json_value(manifest), indent=2), encoding="utf-8")


40428

## 5. Training and checkpoint helpers

Each candidate uses its recorded seed, GELU, dropout 0.10, Adam learning rate `5e-4`, validation every two epochs, `ReduceLROnPlateau`, and the calculated update-budget patience values.


In [5]:
class UniversalBalancedData(pl.LightningDataModule):
    def __init__(self, sampler: ElementStratifiedBatchSampler):
        super().__init__()
        self.train = TensorDataset(torch.as_tensor(universal_train_X), torch.as_tensor(universal_train_y), torch.as_tensor(train_element_codes, dtype=torch.long))
        self.val = TensorDataset(torch.as_tensor(universal_val_X), torch.as_tensor(universal_val_y), torch.as_tensor(val_element_codes, dtype=torch.long))
        self.sampler = sampler

    def train_dataloader(self):
        return DataLoader(self.train, batch_sampler=self.sampler)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=1024, shuffle=False)


class BalancedUniversalModule(PlModule):
    def __init__(self, val_baselines: np.ndarray, **kwargs):
        super().__init__(**kwargs)
        self.register_buffer("val_baselines", torch.as_tensor(val_baselines, dtype=torch.float32))
        self.val_mses, self.val_elements = [], []

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        return self.logged_loss("train_loss", y, self.model(x))

    def on_validation_epoch_start(self):
        self.val_mses, self.val_elements = [], []

    def validation_step(self, batch, batch_idx):
        x, y, elements = batch
        prediction = self.model(x)
        self.val_mses.append(torch.mean((y - prediction) ** 2, dim=1).detach())
        self.val_elements.append(elements.detach())
        return self.logged_loss("val_loss", y, prediction)

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        if not self.val_mses:
            raise RuntimeError("Validation produced no rows")
        mses, elements = torch.cat(self.val_mses), torch.cat(self.val_elements)
        self.log("val_median_mse", torch.quantile(mses, 0.5), on_step=False, on_epoch=True, prog_bar=True)
        relative_medians = []
        for element_index, element in enumerate(ELEMENTS):
            mask = elements == element_index
            if not mask.any():
                raise RuntimeError(f"Validation has no rows for {element}")
            relative_medians.append(torch.quantile(mses[mask], 0.5) / self.val_baselines[element_index])
        self.log("val_balanced_rel_mse", torch.stack(relative_medians).mean(), on_step=False, on_epoch=True, prog_bar=True)


TRAINING_COMPLETE_FILENAME = "TRAINING_COMPLETE.json"


def validate_setting_dir(setting_dir: Path) -> Path:
    setting_dir = setting_dir.resolve()
    if not setting_dir.is_relative_to(RUN_DIR.resolve()):
        raise ValueError(f"Setting directory is outside output run: {setting_dir}")
    if setting_dir.exists() and not setting_dir.is_dir():
        raise FileExistsError(f"Setting path is not a directory: {setting_dir}")
    return setting_dir


def write_setting_settings(setting_dir: Path, settings: dict[str, Any]) -> str:
    value = fingerprint(settings)
    (setting_dir / "settings.json").write_text(json.dumps({"settings": json_value(settings), "fingerprint": value}, indent=2), encoding="utf-8")
    return value


def validate_setting_settings(setting_dir: Path, settings: dict[str, Any]) -> str:
    path = setting_dir / "settings.json"
    if not path.is_file():
        raise RuntimeError(f"Missing settings.json in non-empty candidate directory: {setting_dir}")
    saved = json.loads(path.read_text(encoding="utf-8"))
    value = fingerprint(settings)
    if not isinstance(saved, dict) or saved.get("fingerprint") != value or fingerprint(saved.get("settings")) != value:
        raise RuntimeError(f"Checkpoint settings mismatch: {setting_dir}")
    return value


def best_files(setting_dir: Path) -> list[Path]:
    return sorted(path.resolve() for path in setting_dir.glob("best*.ckpt") if path.is_file())


def validate_completion(setting_dir: Path, settings: dict[str, Any], checkpoint: Path) -> Path:
    marker_path = setting_dir / TRAINING_COMPLETE_FILENAME
    if not marker_path.is_file():
        raise RuntimeError(f"Missing {TRAINING_COMPLETE_FILENAME}: {setting_dir}")
    marker = json.loads(marker_path.read_text(encoding="utf-8"))
    expected = checkpoint.resolve()
    if (marker.get("status"), marker.get("settings_fingerprint"), marker.get("checkpoint"), marker.get("checkpoint_sha256")) != ("complete", fingerprint(settings), str(expected), sha256_file(expected)):
        raise RuntimeError(f"Invalid or mismatched {TRAINING_COMPLETE_FILENAME}: {marker_path}")
    return expected


def best_checkpoint(setting_dir: Path, settings: dict[str, Any]) -> Path | None:
    setting_dir = validate_setting_dir(setting_dir)
    if not setting_dir.exists():
        setting_dir.mkdir(parents=True)
        write_setting_settings(setting_dir, settings)
        return None
    if not any(setting_dir.iterdir()):
        write_setting_settings(setting_dir, settings)
        return None
    validate_setting_settings(setting_dir, settings)
    if not (setting_dir / TRAINING_COMPLETE_FILENAME).is_file():
        raise RuntimeError(f"Non-empty candidate directory is incomplete: {setting_dir}")
    checkpoints = best_files(setting_dir)
    if len(checkpoints) != 1:
        raise RuntimeError(f"Expected exactly one best*.ckpt in {setting_dir}, found {len(checkpoints)}")
    return validate_completion(setting_dir, settings, checkpoints[0])


def finalize_training(setting_dir: Path, settings: dict[str, Any]) -> Path:
    setting_dir = validate_setting_dir(setting_dir)
    value = validate_setting_settings(setting_dir, settings)
    checkpoints = best_files(setting_dir)
    if len(checkpoints) != 1:
        raise RuntimeError(f"Training finished without exactly one best*.ckpt in {setting_dir}, found {len(checkpoints)}")
    checkpoint = checkpoints[0]
    marker = {"status": "complete", "checkpoint": str(checkpoint), "checkpoint_sha256": sha256_file(checkpoint), "settings_fingerprint": value}
    (setting_dir / TRAINING_COMPLETE_FILENAME).write_text(json.dumps(marker, indent=2), encoding="utf-8")
    return checkpoint


def checkpoint_state(checkpoint: Path) -> dict[str, torch.Tensor]:
    payload = torch.load(checkpoint, map_location="cpu")
    state = payload.get("state_dict")
    result = {key.removeprefix("model."): value for key, value in state.items() if key.startswith("model.")} if state else {}
    if not result:
        raise ValueError(f"Checkpoint has no model state: {checkpoint}")
    return result


def load_gelu_model(checkpoint: Path, normalization: str) -> GELUXASBlock:
    model = GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, DROPOUT, normalization)
    model.load_state_dict(checkpoint_state(checkpoint), strict=True)
    return model.to(DEVICE).eval()


def make_callbacks(setting_dir: Path, normalization: str, seed: int, patience: int):
    return [EarlyStopping(monitor=MONITOR, mode="min", patience=patience), ModelCheckpoint(
        dirpath=str(setting_dir), filename=f"best-{normalization}-seed-{seed}-{{epoch:03d}}-{{{MONITOR}:.8f}}", monitor=MONITOR, mode="min", save_top_k=1, save_last=True)]


## 6. Train or safely reuse all 12 normalization-seed candidates

The real sampler is created only after the separate self-checks. Each candidate gets one best checkpoint in a directory that records both normalization and seed.


In [ ]:
candidate_checkpoints: dict[str, Path] = {}
for candidate, settings in zip(candidate_specs, candidate_settings):
    candidate_id = candidate["candidate_id"]
    normalization = candidate["normalization"]
    seed = candidate["seed"]
    candidate_dir = RUN_DIR / candidate_id
    checkpoint = best_checkpoint(candidate_dir, settings)
    if checkpoint is None:
        pl.seed_everything(seed, workers=True)
        sampler = ElementStratifiedBatchSampler(train_indices_by_element, ROWS_PER_ELEMENT, seed)
        module = BalancedUniversalModule(
            val_baselines=val_baselines,
            model=GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, DROPOUT, normalization),
            optimizer=torch.optim.Adam, lr=LEARNING_RATE,
            lr_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
            lr_scheduler_kwargs={"mode": SCHEDULER_MODE, "factor": SCHEDULER_FACTOR, "patience": int(candidate["effective_scheduler_patience"]), "min_lr": MIN_LR},
            lr_scheduler_interval="epoch", lr_scheduler_frequency=SCHEDULER_FREQUENCY, lr_scheduler_monitor=MONITOR,
        )
        trainer = pl.Trainer(
            max_epochs=MAX_EPOCHS, accelerator="auto", devices=1, check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
            callbacks=make_callbacks(candidate_dir, normalization, seed, int(candidate["effective_early_stopping_patience"])),
            logger=CSVLogger(save_dir=str(candidate_dir), name="logs"), enable_progress_bar=True, log_every_n_steps=1,
        )
        trainer.fit(module, datamodule=UniversalBalancedData(sampler))
        checkpoint = finalize_training(candidate_dir, settings)
    else:
        print(f"Reusing completed {candidate_id} candidate: {checkpoint}")
    candidate_checkpoints[candidate_id] = checkpoint
if len(candidate_checkpoints) != EXPECTED_CANDIDATES or set(candidate_checkpoints) != {candidate["candidate_id"] for candidate in candidate_specs}:
    raise AssertionError("Expected one completed checkpoint for each of 12 normalization-seed candidates")


Seed set to 44
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/sweep_balanced_gelu_normalizations_batch128_seeds44_45_46/normalization_BatchNorm1d_seed_44 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type         | Params | Mode  | FLOPs
-------------------------------------------------------
0 | loss  | MSELoss      | 0      | train | 0    
1 | model | GELUXASBlock | 639 K  | train | 0    
-------------------------------------------------------
639 K     Trainable 

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

## 7. Validation-only selection across all 12 candidates

Validation evaluates every normalization-seed candidate on every FEFF dataset. Eta uses that task's training-mean spectrum. The normalization selection and transparent best-seed report occur before any test prediction.


In [ ]:
def predict(model: nn.Module, X: np.ndarray) -> np.ndarray:
    loader = DataLoader(TensorDataset(torch.as_tensor(X, dtype=torch.float32)), batch_size=1024, shuffle=False)
    values = []
    with torch.inference_mode():
        for (batch,) in loader:
            values.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(values)


def split_metrics(split: MLSplits, predictions: np.ndarray, prefix: str) -> dict[str, float]:
    target = getattr(split, prefix).y
    train_mean = split.train.y.mean(axis=0, keepdims=True)
    model_mse = float(np.median(np.mean((target - predictions) ** 2, axis=1)))
    baseline_mse = float(np.median(np.mean((target - train_mean) ** 2, axis=1)))
    if model_mse <= 0 or baseline_mse <= 0:
        raise ValueError(f"Invalid {prefix} MSE: model={model_mse}, baseline={baseline_mse}")
    return {f"{prefix}_median_mse": model_mse, f"{prefix}_baseline_median_mse": baseline_mse, f"{prefix}_eta": baseline_mse / model_mse}


candidate_validation_rows = []
for candidate in candidate_specs:
    candidate_id = candidate["candidate_id"]
    normalization = candidate["normalization"]
    seed = candidate["seed"]
    checkpoint = candidate_checkpoints[candidate_id]
    model = load_gelu_model(checkpoint, normalization)
    for dataset in FEFF_DATASETS:
        row = {"candidate_id": candidate_id, "normalization": normalization, "seed": seed, "batch_size": TOTAL_BATCH_SIZE, "rows_per_element": ROWS_PER_ELEMENT,
               "dataset": dataset, "element": dataset.removesuffix("_FEFF"), "checkpoint": str(checkpoint),
               "checkpoint_sha256": sha256_file(checkpoint), "dropout": DROPOUT, "lr": LEARNING_RATE,
               "effective_scheduler_patience": int(candidate["effective_scheduler_patience"]), "effective_early_stopping_patience": int(candidate["effective_early_stopping_patience"])}
        row.update(split_metrics(splits[dataset], predict(model, splits[dataset].val.X), "val"))
        candidate_validation_rows.append(row)
candidate_validation = pd.DataFrame(candidate_validation_rows).sort_values(["normalization", "seed", "dataset"]).reset_index(drop=True)
if len(candidate_validation) != EXPECTED_DETAILED_ROWS:
    raise AssertionError("Validation output must contain 96 detailed rows: 12 candidates by 8 FEFF tasks")
if candidate_validation["candidate_id"].nunique() != EXPECTED_CANDIDATES:
    raise AssertionError("Validation output must contain all 12 normalization-seed candidates")
candidate_validation.to_csv(RUN_DIR / "candidate_validation.csv", index=False)

candidate_summary = candidate_validation.groupby(["candidate_id", "normalization", "seed"], sort=False, as_index=False).agg(
    checkpoint=("checkpoint", "first"), checkpoint_sha256=("checkpoint_sha256", "first"), batch_size=("batch_size", "first"), rows_per_element=("rows_per_element", "first"),
    mean_val_eta=("val_eta", "mean"), median_val_eta=("val_eta", "median"), mean_val_median_mse=("val_median_mse", "mean"), mean_val_baseline_median_mse=("val_baseline_median_mse", "mean"))
if len(candidate_summary) != EXPECTED_CANDIDATES:
    raise AssertionError("Candidate validation summary must contain 12 normalization-seed rows")

normalization_validation_summary = candidate_summary.groupby("normalization", sort=False, as_index=False).agg(
    seed_mean_val_eta=("mean_val_eta", "mean"), seed_std_val_eta=("mean_val_eta", lambda values: float(np.std(values, ddof=0))), seed_count=("seed", "nunique"))
if len(normalization_validation_summary) != EXPECTED_NORMALIZATION_ROWS:
    raise AssertionError("Normalization validation summary must contain 4 rows")
normalization_validation_summary["seed"] = ",".join(str(seed) for seed in SEEDS)
normalization_validation_summary["selected"] = False
normalization_validation_summary["validation_best_seed"] = pd.NA
max_score = normalization_validation_summary["seed_mean_val_eta"].max()
best_normalizations = normalization_validation_summary[normalization_validation_summary["seed_mean_val_eta"] == max_score]
if len(best_normalizations) != 1:
    raise ValueError(f"Validation normalization selection is not unique: {len(best_normalizations)} normalizations tie")
selected_normalization = str(best_normalizations.iloc[0]["normalization"])
selected_candidates = candidate_summary[candidate_summary["normalization"] == selected_normalization].sort_values(["mean_val_eta", "seed"], ascending=[False, True])
if len(selected_candidates) != len(SEEDS):
    raise AssertionError("Selected normalization must have one validation result for each seed")
validation_best_seed = int(selected_candidates.iloc[0]["seed"])
selected_candidate_id = str(selected_candidates.iloc[0]["candidate_id"])
normalization_validation_summary.loc[normalization_validation_summary["normalization"] == selected_normalization, "selected"] = True
normalization_validation_summary.loc[normalization_validation_summary["normalization"] == selected_normalization, "validation_best_seed"] = validation_best_seed
candidate_summary["selected"] = candidate_summary["normalization"] == selected_normalization
candidate_summary["validation_best_selected_seed"] = candidate_summary["candidate_id"] == selected_candidate_id
candidate_summary["normalization_order"] = candidate_summary["normalization"].map({name: i for i, name in enumerate(NORMALIZATIONS)})
candidate_summary = candidate_summary.sort_values(["normalization_order", "seed"]).drop(columns="normalization_order")
normalization_validation_summary["normalization_order"] = normalization_validation_summary["normalization"].map({name: i for i, name in enumerate(NORMALIZATIONS)})
normalization_validation_summary = normalization_validation_summary.sort_values("normalization_order").drop(columns="normalization_order")
candidate_summary.to_csv(RUN_DIR / "candidate_summary.csv", index=False)
normalization_validation_summary.to_csv(RUN_DIR / "normalization_validation_summary.csv", index=False)
print("=== Detailed validation (96 rows) ===")
display(candidate_validation.style.format({"val_median_mse": "{:.6g}", "val_baseline_median_mse": "{:.6g}", "val_eta": "{:.5f}"}))
print("=== Candidate validation summary (12 normalization-seed rows) ===")
display(candidate_summary.style.format({"mean_val_eta": "{:.5f}", "median_val_eta": "{:.5f}", "mean_val_median_mse": "{:.6g}"}))
print("=== Normalization validation summary (4 rows, selection uses validation only) ===")
display(normalization_validation_summary.style.format({"seed_mean_val_eta": "{:.5f}", "seed_std_val_eta": "{:.5f}"}))
print(f"Selected normalization by mean validation eta across seeds: {selected_normalization}")
print(f"Validation-best seed within selected normalization: {validation_best_seed}")


## 8. All-candidate test evaluation and completion artifacts

Validation selection is fixed before test evaluation. This cell evaluates all 12 completed candidates on all eight FEFF test datasets. Test results do not influence selection. The cell saves and displays detailed and seed-level summaries, then writes hashes and completion metadata.


In [ ]:
# Validation-only selection is fixed above. Test metrics do not select the model.
stale_selected_test = RUN_DIR / "selected_test.csv"
if stale_selected_test.exists():
    stale_selected_test.unlink()
    print(f"Removed stale selected_test.csv from {RUN_DIR}")
selected_checkpoint = candidate_checkpoints[selected_candidate_id]
all_test_rows = []
for candidate in candidate_specs:
    candidate_id = candidate["candidate_id"]
    normalization = candidate["normalization"]
    seed = candidate["seed"]
    checkpoint = candidate_checkpoints[candidate_id]
    model = load_gelu_model(checkpoint, normalization)
    checkpoint_hash = sha256_file(checkpoint)
    for dataset in FEFF_DATASETS:
        row = {"candidate_id": candidate_id, "normalization": normalization, "seed": seed, "dataset": dataset,
               "element": dataset.removesuffix("_FEFF"), "batch_size": TOTAL_BATCH_SIZE, "rows_per_element": ROWS_PER_ELEMENT,
               "checkpoint": str(checkpoint), "checkpoint_sha256": checkpoint_hash, "dropout": DROPOUT, "lr": LEARNING_RATE}
        row.update(split_metrics(splits[dataset], predict(model, splits[dataset].test.X), "test"))
        all_test_rows.append(row)

test_results_all_candidates = pd.DataFrame(all_test_rows).sort_values(["normalization", "seed", "dataset"]).reset_index(drop=True)
if len(test_results_all_candidates) != EXPECTED_DETAILED_ROWS:
    raise AssertionError("Test output must contain 96 detailed rows: 12 candidates by 8 FEFF tasks")
if test_results_all_candidates["candidate_id"].nunique() != EXPECTED_CANDIDATES:
    raise AssertionError("Test output must contain all 12 normalization-seed candidates")
if set(test_results_all_candidates["dataset"]) != set(FEFF_DATASETS):
    raise AssertionError("Test output must include every FEFF task")
test_results_all_candidates.to_csv(RUN_DIR / "test_results_all_candidates.csv", index=False)

test_summary = test_results_all_candidates.groupby(["candidate_id", "normalization", "seed"], sort=False, as_index=False).agg(
    batch_size=("batch_size", "first"), rows_per_element=("rows_per_element", "first"), checkpoint=("checkpoint", "first"), checkpoint_sha256=("checkpoint_sha256", "first"),
    mean_test_median_mse=("test_median_mse", "mean"), mean_test_baseline_median_mse=("test_baseline_median_mse", "mean"), mean_test_eta=("test_eta", "mean"), median_test_eta=("test_eta", "median"))
if len(test_summary) != EXPECTED_CANDIDATES:
    raise AssertionError("Seed-level test summary must contain 12 normalization-seed rows")
test_summary["selected"] = test_summary["candidate_id"] == selected_candidate_id
test_summary["selected_normalization"] = test_summary["normalization"] == selected_normalization
test_summary["validation_best_selected_seed"] = test_summary["candidate_id"] == selected_candidate_id
test_summary["normalization_order"] = test_summary["normalization"].map({name: i for i, name in enumerate(NORMALIZATIONS)})
test_summary = test_summary.sort_values(["normalization_order", "seed"]).drop(columns="normalization_order")
test_summary.to_csv(RUN_DIR / "test_summary.csv", index=False)

normalization_test_summary = test_summary.groupby("normalization", sort=False, as_index=False).agg(
    mean_test_eta=("mean_test_eta", "mean"), std_test_eta=("mean_test_eta", lambda values: float(np.std(values, ddof=0))), seed_count=("seed", "nunique"))
if len(normalization_test_summary) != EXPECTED_NORMALIZATION_ROWS:
    raise AssertionError("Normalization test summary must contain 4 rows")
normalization_test_summary["seed"] = ",".join(str(seed) for seed in SEEDS)
normalization_test_summary["selected"] = normalization_test_summary["normalization"] == selected_normalization
normalization_test_summary["validation_best_selected_seed"] = normalization_test_summary["normalization"] == selected_normalization
normalization_test_summary["validation_best_seed"] = normalization_test_summary["normalization"].map({selected_normalization: validation_best_seed})
normalization_test_summary["normalization_order"] = normalization_test_summary["normalization"].map({name: i for i, name in enumerate(NORMALIZATIONS)})
normalization_test_summary = normalization_test_summary.sort_values("normalization_order").drop(columns="normalization_order")
normalization_test_summary.to_csv(RUN_DIR / "normalization_test_summary.csv", index=False)

evaluation_paths = [RUN_DIR / name for name in ("candidate_validation.csv", "candidate_summary.csv", "normalization_validation_summary.csv", "test_results_all_candidates.csv", "test_summary.csv", "normalization_test_summary.csv")]
manifest.update({
    "status": "complete", "seeds": SEEDS, "candidate_count": EXPECTED_CANDIDATES, "detailed_rows_per_split": EXPECTED_DETAILED_ROWS,
    "selected_normalization": selected_normalization, "validation_best_seed": validation_best_seed, "selected_candidate_id": selected_candidate_id,
    "selected_checkpoint": str(selected_checkpoint), "selected_checkpoint_sha256": sha256_file(selected_checkpoint),
    "candidate_checkpoints": [{"candidate_id": c["candidate_id"], "normalization": c["normalization"], "seed": c["seed"], "checkpoint": str(candidate_checkpoints[c["candidate_id"]]), "checkpoint_sha256": sha256_file(candidate_checkpoints[c["candidate_id"]]), "settings": s} for c, s in zip(candidate_specs, candidate_settings)],
    "evaluation_files": [{"path": str(path), "sha256": sha256_file(path)} for path in evaluation_paths],
    "selection_policy": settings_core["selection_policy"],
    "test_evaluation_policy": "Evaluate all 12 normalization-seed candidates on all eight FEFF test datasets after validation selection is fixed. Test metrics do not select the model.",
    "test_evaluated_for_all_candidates": True, "checkpoint_hashes_recorded": True,
})
manifest_path.write_text(json.dumps(json_value(manifest), indent=2), encoding="utf-8")
run_complete = {"status": "complete", "completion_label": "sweep_balanced_gelu_normalizations_batch128_seeds44_45_46_complete", "settings_manifest": str(manifest_path), "seeds": SEEDS, "candidate_count": EXPECTED_CANDIDATES, "detailed_rows_per_split": EXPECTED_DETAILED_ROWS, "selected_normalization": selected_normalization, "validation_best_seed": validation_best_seed, "selected_candidate_id": selected_candidate_id, "selected_checkpoint": str(selected_checkpoint), "selected_checkpoint_sha256": sha256_file(selected_checkpoint), "selection_policy": settings_core["selection_policy"], "test_evaluation_policy": "All 12 normalization-seed candidates were evaluated on all eight FEFF test datasets after validation selection was fixed. Test metrics did not select the model.", "test_evaluated_for_all_candidates": True, "evaluation_files": manifest["evaluation_files"]}
(RUN_DIR / "RUN_COMPLETE.json").write_text(json.dumps(json_value(run_complete), indent=2), encoding="utf-8")
print("=== Detailed test results (96 rows) ===")
display(test_results_all_candidates)
print("=== Seed-level test summary (12 normalization-seed rows) ===")
display(test_summary)
print("=== Normalization-level test summary (4 rows; no test-based selection) ===")
display(normalization_test_summary)
print(f"Selected normalization by validation only: {selected_normalization}")
print(f"Validation-best seed within selected normalization: {validation_best_seed}")
